Train a Random Forest model to predict California house prices, tune its hyperparameters using GridSearchCV, and track experiments using MLflow.

In [2]:
import pandas as pd
import mlflow
import mlflow.sklearn

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error
from sklearn.datasets import fetch_california_housing

housing = fetch_california_housing()

data = pd.DataFrame(
    housing.data,
    columns=housing.feature_names
)

data["Price"] = housing.target

print("Dataset shape:", data.shape)
data.head(10)

Dataset shape: (20640, 9)


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,Price
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422
5,4.0368,52.0,4.761658,1.103627,413.0,2.139896,37.85,-122.25,2.697
6,3.6591,52.0,4.931907,0.951362,1094.0,2.128405,37.84,-122.25,2.992
7,3.1200,52.0,4.797527,1.061824,1157.0,1.788253,37.84,-122.25,2.414
8,2.0804,42.0,4.294118,1.117647,1206.0,2.026891,37.84,-122.26,2.267
9,3.6912,52.0,4.970588,0.990196,1551.0,2.172269,37.84,-122.25,2.611


Each row represents a California neighbourhood (census block group), and each column describes its housing characteristics.
Column	Meaning
MedInc	Median household income, in units of $10,000
HouseAge	Median age of houses in years
AveRooms	Average rooms per household
AveBedrms	Average bedrooms per household
Population	Number of people living in the area
AveOccup	Average people per household
Latitude / Longitude	Geographic location
Price	Median house value, in units of $100,000

Example: In row 0, Price = 4.526, meaning the median house value is approximately $452,600.

Our goal is to predict Price using the other eight features.

In [3]:
X = data.drop(columns=["Price"])
y = data["Price"]

print("Features:", X.shape)
print("Target:", y.shape)

Features: (20640, 8)
Target: (20640,)


dataset contains 20,640 neighbourhood records, with 8 features used to predict house prices.

Split Data into Training and Testing Sets

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)
print("Training targets:", y_train.shape)
print("Testing targets:", y_test.shape)

Training features: (16512, 8)
Testing features: (4128, 8)
Training targets: (16512,)
Testing targets: (4128,)


We'll use 80% of the data to train our Random Forest model and keep 20% separate to evaluate how accurately it predicts house prices it hasn't seen before.

Define Hyperparameters for Random Forest
Now we'll prepare GridSearchCV, which tests different model settings to find the best combination.

In [4]:
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [5, 10, None],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2]
}

print("Hyperparameter grid created!")

Hyperparameter grid created!


we'll prepare GridSearchCV, which tests different model settings to find the best combination.
GridSearchCV will test 24 combinations, using 3-fold cross-validation.

n_estimators: Number of decision trees.
max_depth: Maximum depth of each tree.
min_samples_split: Minimum samples required to split a node.
min_samples_leaf: Minimum samples in a leaf node.

In [5]:
def hyperparameter_tuning(X_train, y_train, param_grid):
    rf = RandomForestRegressor(random_state=42)

    grid_search = GridSearchCV(
        estimator=rf,
        param_grid=param_grid,
        cv=3,
        n_jobs=-1,
        verbose=2,
        scoring="neg_mean_squared_error"
    )

    grid_search.fit(X_train, y_train)

    return grid_search

print("Hyperparameter tuning function created!")

Hyperparameter tuning function created!


In [6]:
mlflow.set_experiment("house-price-prediction")

with mlflow.start_run(run_name="random-forest-grid-search"):

    grid_search = hyperparameter_tuning(
        X_train, y_train, param_grid
    )

    best_model = grid_search.best_estimator_

    mlflow.log_params(grid_search.best_params_)
    mlflow.log_metric(
        "cv_rmse",
        (-grid_search.best_score_) ** 0.5
    )

    print("Best parameters:", grid_search.best_params_)
    print("Best cross-validation RMSE:",
          (-grid_search.best_score_) ** 0.5)

2026/11/20 06:51:26 INFO mlflow.tracking.fluent: Experiment with name 'house-price-prediction' does not exist. Creating a new experiment.


Fitting 3 folds for each of 24 candidates, totalling 72 fits
[CV] END max_depth=5, min_samples_leaf=1, min_samples_split=2, n_estimators=100; total time=   4.2s
[CV] END max_depth=5, min_samples_leaf=1, min_samples_split=2, n_estimators=100; total time=   4.3s
[CV] END max_depth=5, min_samples_leaf=1, min_samples_split=2, n_estimators=100; total time=   4.6s
[CV] END max_depth=5, min_samples_leaf=1, min_samples_split=2, n_estimators=200; total time=   9.4s
[CV] END max_depth=5, min_samples_leaf=1, min_samples_split=2, n_estimators=200; total time=   8.3s
[CV] END max_depth=5, min_samples_leaf=1, min_samples_split=5, n_estimators=100; total time=   4.3s
[CV] END max_depth=5, min_samples_leaf=1, min_samples_split=2, n_estimators=200; total time=   8.4s
[CV] END max_depth=5, min_samples_leaf=1, min_samples_split=5, n_estimators=100; total time=   4.4s
[CV] END max_depth=5, min_samples_leaf=1, min_samples_split=5, n_estimators=100; total time=   4.5s
[CV] END max_depth=5, min_samples_leaf=

GridSearchCV will train 72 models (24 combinations × 3 folds), select the combination with the lowest prediction error, and log its settings and RMSE in MLflow.

Hyperparameter	Best value
Number of trees	200
Maximum depth	None (unlimited)
Minimum samples per leaf	1
Minimum samples to split	2
Cross-validation RMSE	0.5152

What does this mean?

Your model's RMSE is approximately 0.5152 × $100,000 = $51,523.

This represents the approximate size of its prediction errors during cross-validation. Lower RMSE is better.

Evaluate the Model on Test Data
Now we'll see how accurately your model predicts house prices it hasn't seen before.

Cross-validation helped us select the best model. Now we're evaluating it on the untouched test dataset.

In [7]:
import numpy as np

y_pred = best_model.predict(X_test)

test_rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("Test RMSE:", test_rmse)
print("Approximate error in USD: $", round(test_rmse * 100000, 2))

Test RMSE: 0.5039602414072009
Approximate error in USD: $ 50396.02


Metric	Result
Cross-validation RMSE	0.5152
Test RMSE	0.5040
Approximate test RMSE in dollars	$50,396

What does this mean? Your model's typical prediction error, measured using RMSE, is approximately $50,396. RMSE gives greater weight to larger errors.

Your test performance is slightly better than cross-validation performance, which is encouraging.

In [1]:
from mlflow.models import infer_signature

signature = infer_signature(
    X_test,
    y_pred
)

mlflow.set_experiment("house-price-prediction")

with mlflow.start_run(run_name="best-house-price-model"):
    mlflow.log_params(grid_search.best_params_)
    mlflow.log_metric("test_rmse", float(test_rmse))

    mlflow.sklearn.log_model(
        best_model,
        name="model",
        signature=signature
    )

print("Best house price model saved in MLflow!")

/workspaces/MLIPr/.venv312/lib/python3.12/site-packages/pydantic/_internal/_fields.py:132: UserWarning: Field "model_name" in PromptModelConfig has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
/workspaces/MLIPr/.venv312/lib/python3.12/site-packages/pydantic/_internal/_fields.py:132: UserWarning: Field "model_api" in Guidelines has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(
/workspaces/MLIPr/.venv312/lib/python3.12/site-packages/pydantic/_internal/_fields.py:132: UserWarning: Field "model_api" in ExpectationsGuidelines has conflict with protected namespace "model_".

You may be able to resolve this warning by setting `model_config['protected_namespaces'] = ()`.
  warnings.warn(


NameError: name 'X_test' is not defined

In [5]:
import mlflow

mlflow.set_tracking_uri(
    "sqlite:////workspaces/MLIPr/week-07/mlflow.db"
)

runs = mlflow.search_runs(
    experiment_names=["house-price-prediction"]
)

print(runs[["run_id", "status", "tags.mlflow.runName"]])

                             run_id    status        tags.mlflow.runName
0  f3a7596d4da244ef95d76e8be8c9c7ce   RUNNING     best-house-price-model
1  ae828027bda74cbeb97de8ae22e4c937  FINISHED  random-forest-grid-search


In [6]:
from mlflow.tracking import MlflowClient

client = MlflowClient()

run_id = "f3a7596d4da244ef95d76e8be8c9c7ce"

artifacts = client.list_artifacts(run_id)

print("Saved artifacts:")
for artifact in artifacts:
    print(artifact.path)

Saved artifacts:


In [7]:
from sklearn.ensemble import RandomForestRegressor

best_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42,
    n_jobs=2
)

best_model.fit(X_train, y_train)

print("Best model trained successfully!")

Best model trained successfully!


In [8]:
import joblib

joblib.dump(
    best_model,
    "/workspaces/MLIPr/week-07/house_price_model.joblib",
    compress=3
)

print("Model saved successfully!")

Model saved successfully!


In [9]:
import mlflow

mlflow.set_tracking_uri(
    "sqlite:////workspaces/MLIPr/week-07/mlflow.db"
)

mlflow.set_experiment("house-price-prediction")

with mlflow.start_run(run_name="house-price-saved-model"):
    mlflow.log_artifact(
        "/workspaces/MLIPr/week-07/house_price_model.joblib",
        artifact_path="model"
    )

print("Model file logged successfully in MLflow!")

Model file logged successfully in MLflow!


In [10]:
runs = mlflow.search_runs(
    experiment_names=["house-price-prediction"],
    filter_string="tags.mlflow.runName = 'house-price-saved-model'"
)

run_id = runs.iloc[0]["run_id"]

artifacts = client.list_artifacts(run_id, "model")

print("Saved model files:")
for artifact in artifacts:
    print(artifact.path)

Saved model files:
model/house_price_model.joblib


In [12]:
from mlflow.tracking import MlflowClient

client = MlflowClient()

model_name = "HousePriceRandomForest"

try:
    client.create_registered_model(model_name)
    print("Registered model created!")
except Exception as e:
    if "already exists" in str(e).lower():
        print("Registered model already exists!")
    else:
        raise

Registered model created!


In [13]:
model_version = client.create_model_version(
    name="HousePriceRandomForest",
    source=f"runs:/{run_id}/model/house_price_model.joblib",
    run_id=run_id
)

print("Model:", model_version.name)
print("Version:", model_version.version)

Model: HousePriceRandomForest
Version: 1


In [14]:
model = client.get_model_version(
    name="HousePriceRandomForest",
    version="1"
)

print("Model:", model.name)
print("Version:", model.version)
print("Status:", model.status)
print("Source:", model.source)

Model: HousePriceRandomForest
Version: 1
Status: READY
Source: runs:/9c431ad490f64e3b8bb7d8d3c46a41f9/model/house_price_model.joblib
